# Controlled SLM experiments (Colab T4)
Run cells in order. Each execution of the pipeline cell runs **one** experiment; edit only the four experiment values below for the next run. Set `BRANCH_NAME` once to your own existing branch (never `main`). No cell pushes to GitHub automatically.

Mistral first: 3 epochs / 2e-4, then 5 / 2e-4, 5 / 1e-4, 8 / 1e-4. After that, repeat the first three settings with Llama 3.1 8B. Compare only one change at a time. Llama weights require access approval on Hugging Face. The reference-mode teacher score of 5 is a self-comparison, not an independent quality benchmark.

This notebook does not alter the source YAML. It creates a minimally edited YAML copy and unique output folder on Drive per run. It requires the small T4 memory fixes in `phase1/pipeline.py` (tokenizer-only dataset preparation) and `phase1/finetuning/trainer.py` (FP16 fallback compute).

In [ ]:
# 1. Experiment settings — change these four values between runs.
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
EPOCHS = 3
LEARNING_RATE = 2e-4
RUN_NAME = 'mistral-7b-e3-lr2e4'

# Your existing GitHub branch. Do not use main for experiments.
BRANCH_NAME = 'ak'
REPO_URL = 'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git'
DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
ALLOWED_MODELS = (
    'mistralai/Mistral-7B-Instruct-v0.3',
    'meta-llama/Llama-3.1-8B-Instruct',
)
assert MODEL_ID in ALLOWED_MODELS, f'Choose one of {ALLOWED_MODELS}'
assert isinstance(EPOCHS, int) and EPOCHS > 0
assert 0 < float(LEARNING_RATE) < 1
assert BRANCH_NAME not in ('', 'main', 'master'), 'Use your own non-main branch.'
print('Selected:', MODEL_ID, '| epochs:', EPOCHS, '| LR:', LEARNING_RATE, '| run:', RUN_NAME, '| branch:', BRANCH_NAME)

In [ ]:
# 2. GPU verification. Stop early if Colab did not assign a T4-sized CUDA GPU.
import subprocess, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 2**30
print(f'GPU: {props.name} | VRAM: {vram_gb:.2f} GiB | BF16 supported: {torch.cuda.is_bf16_supported()}')
assert vram_gb >= 14, 'This experiment requires about 15 GiB VRAM; do not run on a smaller GPU.'
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# 3. Mount Drive; source data/checkpoints and experiment results persist here.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
drive_root = Path(DRIVE_ROOT)
assert drive_root.parent.exists(), 'Drive mount failed.'
(drive_root / 'experiments').mkdir(parents=True, exist_ok=True)
print('Drive root:', drive_root)

In [ ]:
# 4. Clone/pull only your selected branch. PAT is passed in process environment,
# not embedded in a URL, command line, notebook output, or .git/config.
import os, base64, subprocess
from urllib.parse import urlsplit
from google.colab import userdata
pat = userdata.get('GITHUB_PAT')
assert pat, 'Add GITHUB_PAT in Colab Secrets and enable notebook access.'
git_env = os.environ.copy()
auth = base64.b64encode(('x-access-token:' + pat).encode()).decode()
git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' + auth, GIT_TERMINAL_PROMPT='0')
del pat, auth
repo_dir = Path('/content/project')
def git(*args, cwd=None, capture=False):
    return subprocess.run(['git', *args], cwd=cwd, env=git_env, check=True,
                          text=True, capture_output=capture)
if repo_dir.exists():
    assert (repo_dir / '.git').is_dir(), '/content/project exists but is not a Git clone.'
    remote = git('remote', 'get-url', 'origin', cwd=repo_dir, capture=True).stdout.strip()
    parsed = urlsplit(remote)
    assert parsed.hostname == 'github.com' and parsed.path.rstrip('/') == urlsplit(REPO_URL).path.rstrip('/'), 'Unexpected origin URL.'
    if remote != REPO_URL: git('remote', 'set-url', 'origin', REPO_URL, cwd=repo_dir)
    assert not git('status', '--porcelain', cwd=repo_dir, capture=True).stdout.strip(), 'Local clone has changes; save them before pulling.'
    git('fetch', 'origin', BRANCH_NAME, cwd=repo_dir)
    git('checkout', BRANCH_NAME, cwd=repo_dir)
    git('pull', '--ff-only', 'origin', BRANCH_NAME, cwd=repo_dir)
else:
    git('clone', '--single-branch', '--branch', BRANCH_NAME, REPO_URL, str(repo_dir))
assert git('branch', '--show-current', cwd=repo_dir, capture=True).stdout.strip() == BRANCH_NAME
code_dir = repo_dir / 'code'
assert (code_dir / 'main.py').is_file(), f'Code directory not found: {code_dir}'
commit_sha = git('rev-parse', 'HEAD', cwd=repo_dir, capture=True).stdout.strip()
print(f'Using branch {BRANCH_NAME} at {commit_sha[:12]} | code: {code_dir}')

In [ ]:
# 5. Same dependency files and order as 00_session_setup.ipynb.
import sys
for filename in ('requirements.txt', 'requirements_colab.txt'):
    req = code_dir / filename
    assert req.is_file(), f'Missing: {req}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=True)
print('Dependencies installed.')

In [ ]:
# 6. Load secrets without displaying their values.
for name in ('ANTHROPIC_API_KEY', 'HF_TOKEN'):
    value = userdata.get(name)
    assert value, f'Add {name} in Colab Secrets and enable notebook access.'
    os.environ[name] = value
optional_openai = userdata.get('OPENAI_API_KEY')
if optional_openai: os.environ['OPENAI_API_KEY'] = optional_openai
os.environ['HF_HOME'] = str(drive_root / 'hf_cache')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('Anthropic and Hugging Face credentials available (values hidden).')

In [ ]:
# 7. Inspect exact config, loader, evaluation artifacts, and model access.
import yaml, hashlib, json, pandas as pd
from huggingface_hub import model_info, hf_hub_download, whoami
source_cfg_path = code_dir / 'configs/phase1_config.yaml'
source_cfg = yaml.safe_load(source_cfg_path.read_text())
assert source_cfg['teacher_llm']['provider'] == 'anthropic' and source_cfg['teacher_llm']['model'] == 'claude-haiku-4-5', 'Teacher changed; recheck comparability.'
for path in ('phase1/finetuning/dataset.py', 'phase1/finetuning/trainer.py',
             'phase1/evaluation/metrics.py', 'phase1/evaluation/llm_judge.py',
             'phase1/evaluation/combine.py', 'phase1/data/schema.py', 'main.py'):
    assert (code_dir / path).is_file(), f'Missing expected implementation: {path}'
pipeline_text = (code_dir / 'phase1/pipeline.py').read_text()
trainer_text = (code_dir / 'phase1/finetuning/trainer.py').read_text()
assert 'Dataset construction needs only the chat template' in pipeline_text, 'Your branch lacks the tokenizer-only T4 memory fix in phase1/pipeline.py. Add/merge the companion fix before running 7B/8B.'
assert 'del ft_model, ft_base, ft_tok' in pipeline_text, 'Your branch still retains a model in VRAM after fine-tuned inference. Add/merge the companion cleanup fix.'
assert 'torch.cuda.is_bf16_supported() else torch.float16' in trainer_text, 'Your branch lacks the FP16 4-bit fallback fix in phase1/finetuning/trainer.py.'
assert 'q_proj' in trainer_text and 'gate_proj' in trainer_text, 'Review LoRA target-module mapping for this branch.'
try:
    info = model_info(MODEL_ID, token=os.environ['HF_TOKEN'])
    print('HF_TOKEN account:', whoami(token=os.environ['HF_TOKEN'])['name'])
    hf_hub_download(repo_id=MODEL_ID, filename='config.json', token=os.environ['HF_TOKEN'], force_download=True)
except Exception as exc:
    raise RuntimeError(f'Cannot download {MODEL_ID}/config.json with HF_TOKEN. For Llama, request/accept access at https://huggingface.co/{MODEL_ID} while signed in as the HF_TOKEN account. Details: {type(exc).__name__}: {exc}') from exc
print('HF model:', info.id, '| gated:', info.gated)
print('Config keys:', {s: list(source_cfg[s]) for s in ('student_slm','training','qlora','lora','evaluation','checkpoints')})
print('Split:', source_cfg['split'], '| top_k:', source_cfg['top_k'], '| seed:', source_cfg['seed'])

### Checkpoint reuse and comparability
The next cell *requires* the prior teacher-labeled CSV and clustered/grouped CSVs. It stops rather than quietly re-calling the teacher or changing the dataset. The source config and CSV fingerprint are compared against earlier successful experiments. Fixed T4 guardrails are batch size 1, accumulation 16, FP16, gradient checkpointing, 4-bit NF4, and max sequence length 1024 for both models; these are recorded with every run. Judge composite on the held-out test split is the primary metric; a score strictly above 4.0 is flagged.

In [ ]:
# 8. Validate checkpoints, then patch only individual YAML lines in a Drive copy.
import re, uuid
from datetime import datetime, timezone
processed = drive_root / 'data/processed'
needed = ('bitext_clustered.csv', 'bitext_grouped.csv', 'bitext_labeled.csv')
missing = [name for name in needed if not (processed / name).is_file()]
assert not missing, f'Missing cached data: {missing}. Run the setup notebook once before experiments.'
labeled_path = processed / 'bitext_labeled.csv'
labeled = pd.read_csv(labeled_path)
teacher_tag = source_cfg['teacher_llm']['model'].replace('/', '_').replace('-', '_').replace('.', '_').replace(' ', '_')
expected = [f'cluster_name_{teacher_tag}_P{i}' for i in range(1, 6)]
assert all(c in labeled.columns for c in expected), f'Cached teacher labels do not match config: {expected}'
assert labeled[expected].notna().all().all(), 'Cached teacher labels contain missing values.'
dataset_hash = hashlib.sha256(labeled_path.read_bytes()).hexdigest()
comparison = {key: source_cfg[key] for key in ('seed','dataset','clustering','top_k','teacher_llm','prompts','split','evaluation')}
comparison['labeled_sha256'] = dataset_hash
comparison_hash = hashlib.sha256(json.dumps(comparison, sort_keys=True).encode()).hexdigest()
results_csv = drive_root / 'experiments/experiment_results.csv'
if results_csv.exists():
    previous = pd.read_csv(results_csv)
    completed = previous[previous['status'] == 'completed']
    assert completed.empty or set(completed['comparison_hash'].dropna()) == {comparison_hash}, 'Dataset/evaluation settings differ from previous experiments. Use a new results series after reviewing the change.'
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', RUN_NAME), 'RUN_NAME must be a short slug (letters/numbers/_/-). '
experiment_id = f'{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{RUN_NAME}_{uuid.uuid4().hex[:8]}'
experiment_dir = drive_root / 'experiments' / experiment_id
experiment_dir.mkdir(parents=True, exist_ok=False)
runs_root = experiment_dir / 'runs'
runs_root.mkdir()
def patch_yaml_line(text, section, key, value):
    lines = text.splitlines(keepends=True)
    section_start = [i for i, line in enumerate(lines) if re.match(r'^' + re.escape(section) + r':(?:\s*#.*)?\s*$', line)]
    assert len(section_start) == 1, f'Expected one YAML section {section}'
    start = section_start[0] + 1
    end = next((i for i in range(start, len(lines)) if re.match(r'^[A-Za-z_][\w-]*:', lines[i])), len(lines))
    hits = [i for i in range(start, end) if re.match(r'^  ' + re.escape(key) + r':', lines[i])]
    assert len(hits) == 1, f'Expected one {section}.{key}'
    i = hits[0]
    match = re.match(r'^(  ' + re.escape(key) + r':\s*)([^#\n]*?)(\s*(?:#.*)?)(\n?)$', lines[i])
    assert match, f'Cannot safely edit {section}.{key}'
    lines[i] = match.group(1) + json.dumps(value) + match.group(3) + match.group(4)
    return ''.join(lines)
overrides = {
    ('student_slm','model_id'): MODEL_ID, ('student_slm','max_seq_length'): 1024,
    ('training','num_train_epochs'): EPOCHS, ('training','learning_rate'): float(LEARNING_RATE),
    ('training','per_device_train_batch_size'): 1, ('training','gradient_accumulation_steps'): 16,
    ('training','gradient_checkpointing'): True, ('training','bf16'): False, ('training','fp16'): True,
    ('qlora','load_in_4bit'): True, ('qlora','use_double_quant'): True,
    ('paths','outputs'): str(runs_root),
    ('pipeline','run_clustering'): False, ('pipeline','run_preprocessing'): False,
    ('pipeline','run_label_generation'): False, ('pipeline','run_finetuning'): True,
    ('pipeline','run_baseline_eval'): True, ('pipeline','run_finetuned_eval'): True,
    ('pipeline','run_llm_judge'): True, ('pipeline','run_business_eval'): True,
    ('evaluation','existing_run_dir'): None, ('evaluation','eval_all_splits'): False,
}
config_text = source_cfg_path.read_text()
for (section, key), value in overrides.items():
    config_text = patch_yaml_line(config_text, section, key, value)
run_cfg = yaml.safe_load(config_text)
for (section, key), value in overrides.items(): assert run_cfg[section][key] == value
run_cfg_path = experiment_dir / 'phase1_config.yaml'
run_cfg_path.write_text(config_text)
manifest = dict(experiment_id=experiment_id, model_id=MODEL_ID, epochs=EPOCHS, learning_rate=LEARNING_RATE,
                run_name=RUN_NAME, branch=BRANCH_NAME, commit_sha=commit_sha, comparison_hash=comparison_hash,
                labeled_sha256=dataset_hash, overrides={f'{s}.{k}':v for (s,k),v in overrides.items()})
(experiment_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Run:', experiment_id, '\nConfig:', run_cfg_path, '\nTeacher calls: 0 (cached)')
print(json.dumps(manifest, indent=2))

In [ ]:
# 9. Memory cleanup in the notebook process. The pipeline runs in a fresh
# subprocess; its existing _clear_device_cache() handles phase transitions.
import gc, torch
gc.collect()
torch.cuda.empty_cache()
free_gb, total_gb = torch.cuda.mem_get_info()
print(f'Free GPU memory before launch: {free_gb/2**30:.2f}/{total_gb/2**30:.2f} GiB')
assert free_gb/2**30 >= 12, 'GPU is already occupied. Restart the runtime or release other models before training.'

In [ ]:
# 10. Run ONE experiment. Do not rerun cells 8–10 for the same settings
# unless you intend to create a new, uniquely identified run.
import time
print('LAUNCH CONFIGURATION')
print(json.dumps({
    'experiment_id':experiment_id, 'branch':BRANCH_NAME, 'commit':commit_sha,
    'model':run_cfg['student_slm'], 'training':{k:run_cfg['training'][k] for k in (
      'num_train_epochs','learning_rate','per_device_train_batch_size',
      'gradient_accumulation_steps','gradient_checkpointing','bf16','fp16')},
    'qlora':run_cfg['qlora'], 'lora':run_cfg['lora'],
    'pipeline':run_cfg['pipeline'], 'split':run_cfg['split'],
    'evaluation':run_cfg['evaluation'], 'outputs':str(runs_root),
    'comparison_hash':comparison_hash
}, indent=2))
launch_marker = experiment_dir / 'launch_started.json'
assert not launch_marker.exists() and not any(runs_root.iterdir()), 'This experiment folder was already launched. Rerun cell 8 to create a fresh run.'
launch_marker.write_text(json.dumps({'experiment_id':experiment_id, 'started_utc':datetime.now(timezone.utc).isoformat()}))
cmd = [sys.executable, 'main.py', '--phase', '1', '--config', str(run_cfg_path), '--device_mode', 'colab']
pipeline_log = experiment_dir / 'pipeline.log'
t0 = time.monotonic()
with pipeline_log.open('w') as log, subprocess.Popen(
    cmd, cwd=code_dir, env=os.environ.copy(), stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
) as proc:
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    exit_code = proc.wait()
wall_time_s = round(time.monotonic() - t0, 1)
gc.collect()
torch.cuda.empty_cache()
log_text = pipeline_log.read_text(errors='replace')
oom = bool(re.search(r'(out of memory|CUDA error: out of memory|CUBLAS_STATUS_ALLOC_FAILED)', log_text, re.I))
print(f'Exit code: {exit_code}; OOM: {oom}; elapsed: {wall_time_s/60:.1f} min; log: {pipeline_log}')
if exit_code and oom:
    print('CUDA OOM: batch size is already 1. Restart the runtime; next try a shorter max_seq_length (e.g. 768) consistently for BOTH models, or use a GPU with more VRAM. Record that change as a new comparison series.')
elif exit_code:
    print('Pipeline failed. Inspect the end of pipeline.log above; results cell records the failure.')

In [ ]:
# 11–12. Extract the held-out TEST metrics and append one persistent row,
# including failure/OOM rows. No score is invented when an artifact is missing.
import csv
run_dirs = [p for p in runs_root.iterdir() if p.is_dir()]
assert len(run_dirs) <= 1, f'Unexpected multiple run directories: {run_dirs}'
pipeline_run_dir = run_dirs[0] if run_dirs else None
eval_dir = pipeline_run_dir / 'evaluation' if pipeline_run_dir else None
def test_value(filename, column):
    path = eval_dir / filename if eval_dir else None
    if not path or not path.exists(): return None
    table = pd.read_csv(path)
    subset = table[(table['split'] == 'test') & (table['model'] == 'finetuned')]
    if subset.empty or column not in subset.columns: return None
    value = pd.to_numeric(subset.iloc[0][column], errors='coerce')
    return float(value) if pd.notna(value) else None
def business_value(metric):
    path = eval_dir / 'business_eval.csv' if eval_dir else None
    if not path or not path.exists(): return None
    table = pd.read_csv(path)
    values = table.loc[table['metric'] == metric, 'value']
    return float(values.iloc[0]) if not values.empty else None
def validation_loss():
    if not pipeline_run_dir: return None
    states = list((pipeline_run_dir / 'models/lora_adapter').glob('checkpoint-*/trainer_state.json'))
    values = []
    for state in states:
        data = json.loads(state.read_text())
        values += [float(item['eval_loss']) for item in data.get('log_history', []) if 'eval_loss' in item]
    return min(values) if values else None
score = test_value('judge_summary.csv', 'composite')
cosine = test_value('metrics_summary.csv', 'cosine_sim_same')
rouge = test_value('metrics_summary.csv', 'rouge_l_same')
split_counts = {part: sum(1 for line in (processed / f'{part}.jsonl').open() if line.strip()) if (processed / f'{part}.jsonl').exists() else 0 for part in ('train','val','test')}
expected_examples = labeled['cluster_id'].nunique() * 5
all_examples_kept = sum(split_counts.values()) == expected_examples
status = 'oom' if oom else ('completed' if exit_code == 0 and score is not None and cosine is not None and rouge is not None and all_examples_kept else ('incomparable' if exit_code == 0 and not all_examples_kept else 'failed'))
row = dict(experiment_id=experiment_id, run_name=RUN_NAME, timestamp_utc=datetime.now(timezone.utc).isoformat(),
           status=status, oom=oom, model_id=MODEL_ID, epochs=EPOCHS, learning_rate=LEARNING_RATE,
           judge_composite_test=score, above_4=(status == 'completed' and score > 4.0),
           cosine_same_test=cosine, rouge_l_same_test=rouge, best_validation_loss=validation_loss(),
           training_time_min=business_value('finetuning_wall_time_min'), wall_time_min=round(wall_time_s/60, 2),
           batch_size=1, gradient_accumulation=16, max_seq_length=1024, qlora_4bit=True,
           train_examples=split_counts['train'], val_examples=split_counts['val'], test_examples=split_counts['test'], expected_examples=expected_examples,
           branch=BRANCH_NAME, commit_sha=commit_sha, comparison_hash=comparison_hash,
           labeled_sha256=dataset_hash, output_dir=str(pipeline_run_dir or ''), log_path=str(pipeline_log))
fields = list(row)
if results_csv.exists():
    with results_csv.open(newline='') as f:
        header = next(csv.reader(f))
    assert header == fields, 'Existing results CSV has a different schema; do not append blindly.'
    assert experiment_id not in set(pd.read_csv(results_csv)['experiment_id']), 'This experiment is already recorded. Do not append it twice.'
with results_csv.open('a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    if f.tell() == 0: writer.writeheader()
    writer.writerow(row)
(experiment_dir / 'result.json').write_text(json.dumps(row, indent=2))
print('Recorded:', row, '\nResults CSV:', results_csv)
if status != 'completed':
    raise RuntimeError(f'Experiment {status}; see {pipeline_log}. If examples were skipped, use a common longer sequence length on a larger GPU; if OOM, use more VRAM or a common shorter length and start a new comparison series.')

In [ ]:
# 13. Comparison table. Judge composite is primary; scores > 4 are flagged.
from IPython.display import display
results = pd.read_csv(results_csv)
columns = ['run_name','model_id','epochs','learning_rate','status','oom',
           'judge_composite_test','above_4','cosine_same_test',
           'rouge_l_same_test','best_validation_loss','training_time_min','output_dir']
display(results[columns].sort_values('judge_composite_test', ascending=False, na_position='last')
        .style.format({'judge_composite_test':'{:.2f}', 'cosine_same_test':'{:.3f}',
                       'rouge_l_same_test':'{:.3f}', 'best_validation_loss':'{:.3f}',
                       'training_time_min':'{:.1f}'}, na_rep='—')
        .highlight_max(subset=['judge_composite_test'], color='#b8e6c4'))
print('Held-out test split; judge target > 4.0. Compare runs sharing the same comparison_hash.')

### 14. Optional demo
For a 7B/8B model on a T4, the repo's live demo loads **both** the base and fine-tuned model simultaneously; it may exceed VRAM. Use a larger GPU for that demo, or make a separate sequential-inference demo later. The training/evaluation run above loads the two models one at a time. The adapter for this experiment is at `pipeline_run_dir / 'models/lora_adapter'`. Do not interpret a single demo cluster as the held-out judge score.

In [ ]:
# 15. Optional Git review. This only displays status and diff, never pushes.
subprocess.run(['git', 'status', '--short', '--branch'], cwd=repo_dir, check=True)
subprocess.run(['git', 'diff', '--stat'], cwd=repo_dir, check=True)
subprocess.run(['git', 'diff'], cwd=repo_dir, check=True)
print('Outputs/configs are on Drive, not in Git. Review before any manual commit/push.')

In [ ]:
# Explicit opt-in only, after reviewing status/diff and committing separately.
PUSH_REVIEWED_COMMIT = False
if PUSH_REVIEWED_COMMIT:
    assert git('branch', '--show-current', cwd=repo_dir, capture=True).stdout.strip() == BRANCH_NAME
    assert not git('status', '--porcelain', cwd=repo_dir, capture=True).stdout.strip(), 'Commit or resolve working-tree changes first.'
    print('Pushing reviewed commit on', BRANCH_NAME)
    git('push', 'origin', BRANCH_NAME, cwd=repo_dir)
else:
    print('No push. Set PUSH_REVIEWED_COMMIT=True only after review and an explicit commit.')

To publish notebook/code changes, explicitly review, stage, and commit them on your branch first. The opt-in push cell defaults to off. Never commit Colab secrets, logs, model weights, or Drive outputs.